# 05 · Modo de operación local del moderador

Este cuaderno publica una página HTML autocontenida en un servidor local. La interfaz usa una sola caja y detecta automáticamente si la entrada es una frase o un enlace de YouTube. El backend ejecuta el mejor modelo clásico, el mejor MiniLM Transformer, el Qwen fine-tuned operativo, la comparación de los tres o su consenso.

Los videos sin subtítulos manuales ni automáticos se rechazan. Los subtítulos se agrupan con ventanas de 30 segundos/600 caracteres, como en preentrenamiento, y cada alerta conserva un enlace temporal. Inferencias y revisiones humanas se guardan localmente por modelo y categoría; las revisiones se exportan a JSONL para reentrenamiento futuro.

La aplicación es de apoyo humano: no autoriza bloqueo, sanción ni moderación autónoma.

In [1]:
from importlib.util import find_spec
import subprocess, sys

DEPENDENCIAS = {
    'yt_dlp': 'yt-dlp>=2025.6',
    'transformers': 'transformers>=4.51,<6',
    'peft': 'peft>=0.15,<1',
    'huggingface_hub': 'huggingface-hub>=0.30',
    'sklearn': 'scikit-learn>=1.4',
}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items() if find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltantes], check=True)
print('Dependencias listas.')

Dependencias listas.


## 1. Verificar artefactos y construir el registro desplegable

Si el checkpoint E5 o los artefactos Qwen sólo están en Drive, se recuperan sin sobrescribir conflictos locales. La selección se rehace únicamente con métricas de validation; test no interviene.

In [2]:
from pathlib import Path
import importlib, json, os, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown

def _locate_project_root_05():
    cwd = Path.cwd().resolve()
    configured = os.environ.get('PLN_PROJECT_ROOT', '').strip()
    candidates = ([Path(configured)] if configured else []) + [
        cwd, *cwd.parents, cwd / 'Trabajo_PLN-MIA-Grupo4',
        cwd / 'trabajo_PLN' / 'Trabajo_PLN-MIA-Grupo4',
        Path('/content/Trabajo_PLN-MIA-Grupo4'),
        Path(r'D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4'),
    ]
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'scripts_auxiliares' / 'crear_bundle_despliegue_05.py').is_file():
            return candidate
    raise FileNotFoundError(
        'No se encontró la raíz del proyecto. Ejecute el cuaderno dentro del repositorio '
        'o defina PLN_PROJECT_ROOT con su ruta.'
    )

ROOT = _locate_project_root_05()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
os.chdir(ROOT)

DRIVE_BUNDLE = Path(r'G:\My Drive\PLN_colab_04_artifacts')
RECOVERY_SCRIPT = ROOT / 'scripts_auxiliares' / 'recuperar_resultados_colab_04_20x.ps1'
DEPLOYMENT_REQUIRED = [
    ROOT / 'modelos/transformer_plano_4/e5_small/best_checkpoint.pt',
    ROOT / 'modelos/transformer_plano_4/e5_small/tokenizer/tokenizer.json',
    ROOT / 'resultados/metricas/qwen3_06b_lora_acoso_amenaza_4/seleccion_operativa_validacion.json',
    ROOT / 'modelos/qwen3_06b_lora_acoso_amenaza_4/epoch_adapters/epoch_03/adapter_model.safetensors',
]
missing_deployment = [path for path in DEPLOYMENT_REQUIRED if not path.is_file()]
if missing_deployment and sys.platform == 'win32' and DRIVE_BUNDLE.is_dir() and RECOVERY_SCRIPT.is_file():
    recovery = subprocess.run([
        'powershell', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(RECOVERY_SCRIPT),
        '-Workspace', str(ROOT), '-DriveBundle', str(DRIVE_BUNDLE), '-DeploymentOnly',
    ], text=True, capture_output=True)
    if recovery.returncode != 0:
        raise RuntimeError(recovery.stderr or recovery.stdout)
    print(recovery.stdout.strip())
missing_deployment = [str(path.relative_to(ROOT)) for path in DEPLOYMENT_REQUIRED if not path.is_file()]
if missing_deployment:
    raise FileNotFoundError('Faltan artefactos de operación:\n' + '\n'.join(missing_deployment))

from scripts_auxiliares import registro_modelos_produccion_4 as registry4
from scripts_auxiliares import servidor_moderacion_05 as moderation05
REGISTRY = registry4.build_registry()
SERVICE = moderation05.ModerationService(REGISTRY)
display(pd.DataFrame([
    {'tipo': slot, 'modelo': model['label'], 'PR-AUC validation': model['validation_damage_pr_auc_macro'], 'test_usado_para_seleccion': model['test_used_for_selection']}
    for slot, model in REGISTRY['models'].items()
]))

,tipo,modelo,PR-AUC validation,test_usado_para_seleccion
0,classical,SVM lineal calibrado palabra+carácter · Plano,0.469987,False
1,transformer,Multilingual E5-small (linaje MiniLM),0.504120,False
2,qwen,Qwen3-0.6B LoRA · época operativa 3,0.542496,False


## 2. Crear `05_frontend_despliegue`

Esta etapa crea una carpeta portable con HTML, backend, base SQLite nueva, registro y hashes, los tres modelos seleccionados, Qwen base completo y archivos Docker. Requiere la ejecución final en orden `04_207 → 04_208`; se detiene si la auditoría no corresponde a la selección Qwen actual o conserva pendientes. No copia estadísticas ni revisiones humanas existentes. El consenso incluido es mayoritario: **2 de 3**, no unanimidad.

In [3]:
# Parámetros de empaquetado. El resultado ocupa aproximadamente 1.7 GiB.
CREAR_CARPETA_DESPLIEGUE = True
RECREAR_CARPETA_DESPLIEGUE = True
DESCARGAR_QWEN_BASE_SI_FALTA = True

# Esta celda también puede ejecutarse de forma aislada: localiza el repositorio
# y reconstruye el registro si la sección 1 todavía no se ejecutó.
if '_locate_project_root_05' not in globals():
    from pathlib import Path
    import importlib, os, sys
    def _locate_project_root_05():
        cwd = Path.cwd().resolve()
        configured = os.environ.get('PLN_PROJECT_ROOT', '').strip()
        candidates = ([Path(configured)] if configured else []) + [
            cwd, *cwd.parents, cwd / 'Trabajo_PLN-MIA-Grupo4',
            cwd / 'trabajo_PLN' / 'Trabajo_PLN-MIA-Grupo4',
            Path('/content/Trabajo_PLN-MIA-Grupo4'),
            Path(r'D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4'),
        ]
        for candidate in dict.fromkeys(path.expanduser().resolve() for path in candidates):
            if (candidate / 'scripts_auxiliares' / 'crear_bundle_despliegue_05.py').is_file():
                return candidate
        raise FileNotFoundError(
            'No se encontró la raíz del proyecto. Defina PLN_PROJECT_ROOT con su ruta.'
        )
ROOT = _locate_project_root_05()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
os.chdir(ROOT)
from IPython.display import display

if 'REGISTRY' not in globals():
    from scripts_auxiliares import registro_modelos_produccion_4 as registry4
    REGISTRY = registry4.build_registry()

if CREAR_CARPETA_DESPLIEGUE:
    from scripts_auxiliares import crear_bundle_despliegue_05 as bundle05
    BUNDLE_05 = bundle05.build_deployment_bundle(
        REGISTRY,
        recreate=RECREAR_CARPETA_DESPLIEGUE,
        download_qwen_base_if_needed=DESCARGAR_QWEN_BASE_SI_FALTA,
    )
    display({
        'carpeta': BUNDLE_05['output_dir'],
        'tamaño_GiB': round(BUNDLE_05['total_gib'], 3),
        'archivos': len(BUNDLE_05['files']),
        'consenso': BUNDLE_05['consensus'],
        'modelos': BUNDLE_05['models'],
    })
else:
    print('Creación del bundle omitida por configuración.')

{'carpeta': 'D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\05_frontend_despliegue',
 'tamaño_GiB': 1.6,
 'archivos': 51,
 'consenso': {'minimum_votes': 2,
  'total_models': 3,
  'unanimity_required': False},
 'modelos': {'classical': 'SVM lineal calibrado palabra+carácter · Plano',
  'transformer': 'Multilingual E5-small (linaje MiniLM)',
  'qwen': 'Qwen3-0.6B LoRA · época operativa 3'}}

## 3. Ejecutar una entrada directamente en el cuaderno

La celda siguiente contiene todas las variables de operación. `TIPO_ENTRADA='auto'` reconoce YouTube o texto igual que la página. Cambie `ENTRADA` y ejecútela; no es necesario iniciar el servidor.

In [12]:
# Parámetros de ejecución directa (todos están deliberadamente en esta celda).
ENTRADA = 'https://www.youtube.com/watch?v=otL1vletFgM'  # frase o enlace de YouTube
TIPO_ENTRADA = 'auto'  # auto | text | youtube
MODO_MODELO = 'consensus'  # classical | transformer | qwen | compare | consensus
IDIOMAS_SUBTITULOS = ('es', 'es-419', 'es-US', 'en')
MAX_CHUNKS = 300
GUARDAR_ESTADISTICAS = True

if ENTRADA.strip():
    RESULTADO = SERVICE.analyze(
        ENTRADA,
        mode=MODO_MODELO,
        input_type=TIPO_ENTRADA,
        subtitle_languages=IDIOMAS_SUBTITULOS,
        max_chunks=MAX_CHUNKS,
        persist=GUARDAR_ESTADISTICAS,
    )
    display({key: RESULTADO[key] for key in ('analysis_id', 'input_type', 'mode', 'summary')})
    display(pd.DataFrame([
        {
            'chunk_id': chunk['chunk_id'], 'inicio': chunk['start_seconds'],
            'modelo': result['model_label'], 'etiquetas': result['predicted_labels'],
            'confianza': result['confidence'], 'requiere_revision': result['requires_review'],
            'enlace': chunk['watch_url'],
        }
        for chunk in RESULTADO['chunks'] for result in chunk['results']
    ]))
else:
    print('Defina ENTRADA para ejecutar inferencia directa.')

{'analysis_id': '4b9a7ba3-a0e4-4eea-af1b-54a6ea9bbd43',
 'input_type': 'youtube',
 'mode': 'consensus',
 'summary': {'chunks': 56,
  'alert_chunks': 28,
  'models_executed': ['classical', 'transformer', 'qwen'],
  'persisted': True}}

,chunk_id,inicio,modelo,etiquetas,confianza,requiere_revision,enlace
0,otL1vletFgM_0000,0.16,SVM lineal calibrado palabra+carácter · Plano,[SEGURO],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=0s
1,otL1vletFgM_0000,0.16,Multilingual E5-small (linaje MiniLM),[SEGURO],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=0s
2,otL1vletFgM_0000,0.16,Qwen3-0.6B LoRA · época operativa 3,[SEGURO],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=0s
3,otL1vletFgM_0000,0.16,Consenso mayoritario de los tres modelos,[SEGURO],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=0s
4,otL1vletFgM_0001,29.04,SVM lineal calibrado palabra+carácter · Plano,[SEGURO],alta,False,https://www.youtube.com/watch?v=otL1vletFgM&t=29s
...,...,...,...,...,...,...,...
219,otL1vletFgM_0054,1569.24,Consenso mayoritario de los tres modelos,[SEGURO],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=...
220,otL1vletFgM_0055,1597.88,SVM lineal calibrado palabra+carácter · Plano,[SEGURO],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=...
221,otL1vletFgM_0055,1597.88,Multilingual E5-small (linaje MiniLM),[ACOSO_AMENAZA],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=...
222,otL1vletFgM_0055,1597.88,Qwen3-0.6B LoRA · época operativa 3,[SEGURO],alta,True,https://www.youtube.com/watch?v=otL1vletFgM&t=...


## 4. Iniciar la página HTML local

El servidor escucha sólo en `127.0.0.1` por defecto. El HTML, CSS, JavaScript y ayuda están en un único archivo; la inferencia se realiza en el backend local con los checkpoints verificados.

In [ ]:
HOST = '127.0.0.1'
PORT = 8765  # use 0 para elegir un puerto libre
ABRIR_NAVEGADOR = True
PERMITIR_RED = False

if 'SERVER_05' in globals():
    SERVER_05.stop()
SERVER_05 = moderation05.start_server(
    SERVICE, host=HOST, port=PORT, open_browser=ABRIR_NAVEGADOR, allow_network=PERMITIR_RED
)
display(Markdown(f'Aplicación disponible en **[{SERVER_05.url}]({SERVER_05.url})**'))
print('HTML:', moderation05.HTML_PATH.relative_to(ROOT))
print('SQLite:', moderation05.DATABASE_PATH.relative_to(ROOT))
print('Revisiones para reentrenamiento:', moderation05.RETRAINING_JSONL_PATH.relative_to(ROOT))

Aplicación disponible en **[http://127.0.0.1:8765/](http://127.0.0.1:8765/)**

HTML: Cuadernos\frontend\produccion_moderador.html
SQLite: resultados\operacion_05\estadisticas_moderacion.sqlite3
Revisiones para reentrenamiento: resultados\operacion_05\revisiones_para_reentrenamiento.jsonl


[05] 127.0.0.1 - "GET / HTTP/1.1" 200 -
[05] 127.0.0.1 - "GET /favicon.ico HTTP/1.1" 404 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -
[05] 127.0.0.1 - "POST /api/analyze HTTP/1.1" 200 -


## 5. Detener el servidor

Ejecute esta celda antes de cerrar el kernel si quiere liberar el puerto inmediatamente.

In [14]:
if 'SERVER_05' in globals():
    SERVER_05.stop()
    del SERVER_05
    print('Servidor 05 detenido.')
else:
    print('No hay servidor 05 activo.')

Servidor 05 detenido.


## Documentación

La guía completa de modos, revisión, estadísticas, reentrenamiento y límites está en `Cuadernos/05_MODO_OPERACION.md`. La misma ayuda resumida está disponible dentro de la página.